# Internal Bar Strength y medias exponenciales

Backtest diario de una estrategia long/cash sobre AAPL. Abre una posición cuando IBS es como máximo 0,25 y la EMA de 8 sesiones está por debajo de la de 24. Cierra cuando IBS es como mínimo 0,75 y la EMA de 8 está por encima de la de 24.

El ejemplo descarga precios OHLC de Yahoo Finance con `yfinance`. Las señales se calculan al cierre y se simulan a ese mismo precio. El modelo usa acciones fraccionarias, cierra cualquier posición al final de la muestra y omite comisiones y deslizamiento. Los importes se expresan en dólares estadounidenses.

In [ ]:
import yfinance as yf


def get_aapl_data(start_date, end_date, interval="1d"):
    data = yf.download(
        "AAPL",
        start=start_date,
        end=end_date,
        interval=interval,
        auto_adjust=False,
        multi_level_index=False,
        progress=False,
    )
    if data is None or data.empty:
        raise RuntimeError("Yahoo Finance no devolvió datos para AAPL.")
    return data.dropna(subset=["High", "Low", "Close"])


def calculate_ibs(data):
    price_range = data["High"] - data["Low"]
    return (data["Close"] - data["Low"]).div(price_range.where(price_range > 0))


def calculate_ema(data, window):
    return data["Close"].ewm(span=window, adjust=False).mean()


def backtest_strategy(data, initial_capital=2000):
    capital = initial_capital
    shares = 0.0
    trade_cost = 0.0
    trades = []

    for row in data.itertuples():
        if row.Buy_Signal and shares == 0:
            trade_cost = capital
            shares = capital / row.Close
            capital = 0.0
            trades.append({"Type": "Buy", "Date": row.Index, "Price": row.Close})
        elif row.Sell_Signal and shares > 0:
            capital = shares * row.Close
            shares = 0.0
            trades.append({
                "Type": "Sell",
                "Date": row.Index,
                "Price": row.Close,
                "Gain/Loss": capital - trade_cost,
            })

    if shares > 0:
        close = data["Close"].iloc[-1]
        capital = shares * close
        trades.append({
            "Type": "Sell",
            "Date": data.index[-1],
            "Price": close,
            "Gain/Loss": capital - trade_cost,
        })

    return capital - initial_capital, trades

In [ ]:
start_date = "2011-01-01"
end_date = "2023-08-06"
initial_capital = 2000

data = get_aapl_data(start_date, end_date)
data["IBS"] = calculate_ibs(data)
data["EMA_8"] = calculate_ema(data, window=8)
data["EMA_24"] = calculate_ema(data, window=24)
data["Buy_Signal"] = (data["IBS"] <= 0.25) & (data["EMA_8"] < data["EMA_24"])
data["Sell_Signal"] = (data["IBS"] >= 0.75) & (data["EMA_8"] > data["EMA_24"])

total_gain_loss, trades = backtest_strategy(data, initial_capital)
print(f"Resultado total: {total_gain_loss:,.2f} USD")
print("\nOperaciones:")
for trade in trades:
    details = f"{trade['Type']} | {trade['Date']:%Y-%m-%d} | {trade['Price']:.2f} USD"
    if "Gain/Loss" in trade:
        details += f" | Resultado: {trade['Gain/Loss']:,.2f} USD"
    print(details)